# `data_utils.py` — Dataset Classes & Data Loaders

## Purpose
Provides all PyTorch `Dataset` and `DataLoader` components needed to train and evaluate the multi-aspect sentiment model.

## Key Concepts

### One sample = (text, aspect) pair
A single CSV row with N aspect labels becomes N training samples. For example, if a review has scores for `colour`, `smell`, and `price`, that becomes 3 separate samples in the dataset:
```
('Great shade, smells weird, overpriced', 'colour', label=positive)
('Great shade, smells weird, overpriced', 'smell',  label=negative)
('Great shade, smells weird, overpriced', 'price',  label=negative)
```

### Dependency Parsing (optional)
`DependencyParsingDataset` extends the base dataset by pre-computing the spaCy dependency parse tree for each unique text. These trees are passed to the GCN as `edge_index` tensors.

## Classes & Functions

| Name | Type | Description |
|------|------|-------------|
| `CosmeticReviewDataset` | Dataset | Base dataset — tokenises text, returns (input_ids, aspect_id, label) |
| `DependencyParsingDataset` | Dataset | Extends base — adds pre-computed dependency parse trees |
| `DependencyParser` | class | spaCy wrapper for parsing text into dependency edge_index |
| `collate_fn_with_dependencies` | function | Custom batch collator that handles variable-size edge_index lists |
| `create_dataloaders()` | function | One-call factory to build train/val/test loaders |
| `compute_class_weights()` | function | Returns per-aspect class counts for initialising HybridLoss |

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer  # Handles BPE tokenisation for RoBERTa
import re
from collections import Counter

## 1. Base Dataset — `CosmeticReviewDataset`
Loads a CSV file and expands each row into one sample per aspect. Tokenises the text with the RoBERTa tokeniser.

In [ ]:
class CosmeticReviewDataset(Dataset):
    def __init__(self, data_path, tokenizer, config, aspect_names, is_train=True):
        self.data         = pd.read_csv(data_path)   # Load the CSV
        self.tokenizer    = tokenizer
        self.config       = config
        self.aspect_names = aspect_names
        self.is_train     = is_train
        self.max_length   = config['data']['max_seq_length']   # Max token length (usually 128)
        self.text_column  = config['data']['text_column']      # Column name in CSV ('data' or 'text_clean')
        self.label_map    = config['aspects']['label_map']     # {'negative': 0, 'neutral': 1, 'positive': 2}

        self.samples = self._prepare_samples()  # Expand rows → (text, aspect, label) triples
        print(f'Loaded {len(self.samples)} samples from {data_path}')
        self._print_statistics()

    def _prepare_samples(self):
        """Convert each labelled (row, aspect) pair into a sample dict."""
        samples = []
        for idx, row in self.data.iterrows():
            text = str(row[self.text_column]) if pd.notna(row[self.text_column]) else ''
            if not text.strip(): continue   # Skip empty reviews

            for aspect in self.aspect_names:
                if pd.notna(row[aspect]):    # Only include aspects that have a label
                    label_str = str(row[aspect]).lower()
                    if label_str in self.label_map:
                        samples.append({
                            'text'        : text,
                            'aspect'      : aspect,
                            'aspect_id'   : self.aspect_names.index(aspect),  # Integer index for embedding lookup
                            'label'       : self.label_map[label_str],         # Integer class label
                            'original_idx': idx,
                        })
        return samples

    def _print_statistics(self):
        """Log aspect and label counts — useful for diagnosing imbalance."""
        aspect_counts = Counter([s['aspect'] for s in self.samples])
        label_names   = {v: k for k, v in self.label_map.items()}
        label_counts  = Counter([s['label']  for s in self.samples])
        print('Aspect distribution:', dict(aspect_counts))
        print('Label distribution: ', {label_names[k]: v for k, v in label_counts.items()})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample   = self.samples[idx]
        encoding = self.tokenizer(
            sample['text'],
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',   # Pad all sequences to max_length for batch uniformity
            truncation=True,         # Truncate sequences longer than max_length
            return_tensors='pt'      # Return PyTorch tensors
        )
        return {
            'input_ids'    : encoding['input_ids'].squeeze(0),       # (max_length,)
            'attention_mask': encoding['attention_mask'].squeeze(0), # (max_length,) — 0 for padding
            'aspect_id'    : torch.tensor(sample['aspect_id'], dtype=torch.long),
            'label'        : torch.tensor(sample['label'],     dtype=torch.long),
            'text'         : sample['text'],    # Kept as string for explainability
            'aspect'       : sample['aspect'],
            'review_id'    : sample['original_idx'],
        }

## 2. Dependency Parsing Dataset
Extends `CosmeticReviewDataset` with pre-computed spaCy dependency trees.

**Important edge pruning**: spaCy uses word-level indices, but RoBERTa uses subword (BPE) indices capped at `max_length`. We prune edges whose node indices exceed `max_length` to prevent `IndexError` in the GCN's `scatter_add_`.

In [ ]:
class DependencyParsingDataset(CosmeticReviewDataset):
    def __init__(self, data_path, tokenizer, config, aspect_names, dependency_parser=None, is_train=True):
        super().__init__(data_path, tokenizer, config, aspect_names, is_train)
        self.dependency_parser = dependency_parser

        if dependency_parser is not None:
            print('Pre-computing dependency trees...')
            self.dependency_trees = self._compute_dependency_trees()  # One-time upfront parse
        else:
            self.dependency_trees = None

    def _compute_dependency_trees(self):
        """Pre-parse all unique texts and store (tokens, edge_index, edge_types) per text."""
        from tqdm import tqdm
        unique_texts = list(set(s['text'] for s in self.samples))  # Avoid re-parsing duplicates
        trees = {}
        for text in tqdm(unique_texts, desc='Parsing dependencies'):
            try:
                toks, ei, et = self.dependency_parser.parse(text)
                trees[text]  = {'tokens': toks, 'edge_index': ei, 'edge_types': et}
            except Exception as e:
                print(f'Error parsing: {e}')
                trees[text] = {'tokens': [], 'edge_index': torch.zeros((2, 0), dtype=torch.long), 'edge_types': []}
        return trees

    def __getitem__(self, idx):
        item = super().__getitem__(idx)  # Get base item (input_ids, label, etc.)

        if self.dependency_trees is not None:
            dep_info   = self.dependency_trees.get(item['text'], {})
            edge_index = dep_info.get('edge_index', torch.zeros((2, 0), dtype=torch.long))

            # Prune edges with indices beyond max_length to avoid out-of-bounds in GCN scatter_add_
            if edge_index.size(1) > 0:
                mask       = (edge_index[0] < self.max_length) & (edge_index[1] < self.max_length)
                edge_index = edge_index[:, mask]

            item['edge_index'] = edge_index
            item['tokens']     = dep_info.get('tokens', [])
            item['edge_types'] = dep_info.get('edge_types', [])
        return item

## 3. Custom Collator
A standard DataLoader collator can't handle lists of tensors with variable edge_index sizes. This custom function pads/stacks the fixed-size tensors and keeps dependency info as lists.

In [ ]:
def collate_fn_with_dependencies(batch):
    """Custom collate function to handle variable-size edge_index per sample."""
    return {
        'input_ids'    : torch.stack([item['input_ids']    for item in batch]),  # (B, seq_len)
        'attention_mask': torch.stack([item['attention_mask'] for item in batch]),
        'aspect_ids'   : torch.stack([item['aspect_id']   for item in batch]),  # (B,)
        'labels'       : torch.stack([item['label']        for item in batch]),  # (B,)
        'review_ids'   : [item['review_id'] for item in batch],
        'texts'        : [item['text']      for item in batch],
        'aspects'      : [item['aspect']    for item in batch],
        # Dependency info: list of (2, num_edges) tensors — kept as list since sizes differ per sample
        'edge_indices' : [item.get('edge_index') for item in batch],
        'tokens'       : [item.get('tokens', []) for item in batch],
        'edge_types'   : [item.get('edge_types', []) for item in batch],
    }

## 4. DependencyParser — spaCy Wrapper

In [ ]:
class DependencyParser:
    """Wraps spaCy to parse text and return dependency edges as a PyTorch edge_index tensor."""

    def __init__(self, model_name='en_core_web_sm'):
        import spacy
        try:
            self.nlp = spacy.load(model_name)
        except OSError:
            print(f'spaCy model {model_name} not found — installing...')
            import subprocess; subprocess.run(['python', '-m', 'spacy', 'download', model_name])
            self.nlp = spacy.load(model_name)

    def parse(self, text):
        """
        Args:
            text: Input review string
        Returns:
            tokens:      List of token strings
            edge_index:  (2, num_edges) LongTensor of [head, dependent] directed edges
            edge_types:  List of dependency label strings (e.g. 'nsubj', 'dobj')
        """
        doc        = self.nlp(text)
        tokens     = [token.text for token in doc]
        edges      = []
        edge_types = []

        for token in doc:
            if token.head != token:  # Exclude self-loops (root has head == self)
                edges.append([token.head.i, token.i])  # Directed edge: head → dependent
                edge_types.append(token.dep_)            # Dependency relation label

        edge_index = torch.tensor(edges, dtype=torch.long).t() if edges else torch.zeros((2, 0), dtype=torch.long)
        return tokens, edge_index, edge_types

## 5. Factory Functions

In [ ]:
def create_dataloaders(config, tokenizer, dependency_parser=None):
    """Build train/val/test DataLoaders from config. Handles both dataset modes."""
    data_config  = config['data']
    hw_config    = config['hardware']
    aspect_names = config['aspects']['names']

    # Choose the right dataset class based on whether dependency parsing is enabled
    if data_config.get('use_dependency_parsing', False) and dependency_parser is not None:
        DatasetClass = DependencyParsingDataset
        extra = {'dependency_parser': dependency_parser}
    else:
        DatasetClass = CosmeticReviewDataset
        extra = {}

    def make_loader(path, is_train):
        ds = DatasetClass(path, tokenizer, config, aspect_names, is_train=is_train, **extra)
        return DataLoader(ds,
            batch_size=config['training']['batch_size'],
            shuffle=is_train,                    # Only shuffle training data
            num_workers=hw_config['num_workers'],
            pin_memory=hw_config['pin_memory'],  # GPU transfer optimisation
            collate_fn=collate_fn_with_dependencies
        )

    return (
        make_loader(data_config['train_path'], True),
        make_loader(data_config['val_path'],   False),
        make_loader(data_config['test_path'],  False),
    )


def compute_class_weights(data_path, aspect_names, label_map):
    """Returns {aspect: [neg_count, neu_count, pos_count]} for HybridLoss init."""
    df = pd.read_csv(data_path)
    return {
        aspect: [int((df[aspect] == lbl_str).sum()) for lbl_str, _ in sorted(label_map.items(), key=lambda x: x[1])]
        for aspect in aspect_names
    }